In [1]:
# Standard libraries
import os
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, mutual_info_regression, f_classif, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# Other utilities
from scipy.stats import randint, uniform
import joblib

# Standard libraries
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.impute import SimpleImputer

In [12]:
path = "../../data/processed/"
sites = pd.read_parquet(os.path.join(path, "dep_codes.parquet"))
regiones = sites['HERlvl1Code'].drop_duplicates().tolist()
len(regiones)

22

In [22]:
path = "../../notebooks/06_cb_regression/dfs_tp_dated/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

from types import SimpleNamespace

# Después de llenar `dfs`:
d = SimpleNamespace(**dfs)

Loaded df_1 with shape (1485, 110)
Loaded df_10 with shape (3926, 149)
Loaded df_11 with shape (1289, 164)
Loaded df_12 with shape (4677, 177)
Loaded df_13 with shape (1003, 173)
Loaded df_14 with shape (7742, 162)
Loaded df_15 with shape (1223, 161)
Loaded df_16 with shape (379, 114)
Loaded df_17 with shape (803, 152)
Loaded df_18 with shape (1164, 153)
Loaded df_19 with shape (500, 135)
Loaded df_2 with shape (352, 98)
Loaded df_20 with shape (508, 210)
Loaded df_21 with shape (2663, 185)
Loaded df_22 with shape (152, 177)
Loaded df_3 with shape (4996, 142)
Loaded df_4 with shape (596, 142)
Loaded df_5 with shape (2313, 132)
Loaded df_6 with shape (2134, 143)
Loaded df_7 with shape (524, 108)
Loaded df_8 with shape (548, 124)
Loaded df_9 with shape (10254, 170)


In [23]:
d.df_1

,SamplingOperations_code,TotalAbundance_SamplingOperation,Achaf02,Achat02,Achde03,Acheu01,Achho03,Achla02,Achli03,Achmi02,...,OrganicMicropollutants_Status180D,OrganicMicropollutants_Status90D,MineralMicropollutants_Status1Y,MineralMicropollutants_Status180D,MineralMicropollutants_Status90D,Uncommon_Taxons,IBD,IBD_EQR,IBD_EQR_Status,Date_SamplingOperation
0,S05168100_20070809,400,NaN,45.000000,NaN,NaN,NaN,NaN,NaN,80.000000,...,High,High,None,None,None,0.000000,20.0,1.0,High,2007-08-09
1,S05168100_20080806,400,NaN,60.000000,NaN,NaN,NaN,NaN,NaN,25.000000,...,None,None,None,None,None,7.500000,20.0,1.0,High,2008-08-06
2,S05168100_20090625,400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,...,Good,Good,Moderate,Moderate,Moderate,22.500000,20.0,1.0,High,2009-06-25
3,S05168100_20100629,414,NaN,24.154589,NaN,NaN,NaN,NaN,NaN,36.231884,...,None,None,Poor,None,None,0.000000,20.0,1.0,High,2010-06-29
4,S05168100_20110809,400,NaN,2.500000,NaN,NaN,NaN,NaN,NaN,90.000000,...,None,None,None,None,None,10.000000,20.0,1.0,High,2011-08-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1480,S06175540_20150728,407,NaN,34.398034,NaN,NaN,NaN,NaN,NaN,27.027027,...,None,None,Good,None,None,31.941032,NaN,NaN,None,2015-07-28
1481,S06175600_20170904,400,NaN,NaN,NaN,NaN,NaN,NaN,10.00000,5.000000,...,None,None,Poor,None,None,172.500000,NaN,NaN,None,2017-09-04
1482,S06175600_20180926,411,NaN,301.703163,NaN,NaN,9.73236,NaN,26.76399,48.661800,...,Moderate,Moderate,Good,Good,Good,2.433090,NaN,NaN,None,2018-09-26
1483,S06175645_20220809,418,NaN,47.846890,502.392344,4.784689,NaN,NaN,NaN,33.492823,...,Moderate,Moderate,Moderate,Moderate,Moderate,4.784689,NaN,NaN,None,2022-08-09


In [19]:
path = "../../data/raw/"
pressures = pd.read_parquet(os.path.join(path, "04_PressureStatus_GTstudentproject_B.parquet"))
pressures
dates = pressures [['SamplingOperations_code','Date_SamplingOperation']]